# The 12 Math Ideas You’ll See in Most Deep Learning Papers

This notebook is a **notation-first survival guide**. It explains 12 recurring mathematical ideas in deep learning papers, with **clean LaTeX** and short, runnable code snippets.

**How to use:** Read the math, then run the code cell under each concept (optional). Your goal is to make the symbols feel *normal*.


## 0. Quick Notation Legend

You’ll see these everywhere:

- $x$ : input (vector, image, token embedding)  
- $y$ : target / label  
- $\theta$ : all model parameters (weights and biases)  
- $W$ : weight matrix, $b$ : bias vector  
- $L(\theta)$ : loss (objective)  
- $\nabla_\theta L$ : gradient of the loss w.r.t. parameters  
- $p(\cdot)$ : probability distribution  


In [ ]:
import torch
import torch.nn.functional as F
print('PyTorch version:', torch.__version__)


---

## 1) Vectors and Feature Representation

A vector is an ordered list of numbers:

$$
x = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_d \end{bmatrix} \in \mathbb{R}^d
$$

**Meaning:** $x$ has $d$ features (pixels, embedding dims, sensor values, etc.).


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print('x =', x)
print('shape:', x.shape)


---

## 2) Matrices and Linear Transformations

A matrix is a 2D array of numbers:

$$
W \in \mathbb{R}^{m \times d}
$$

It represents a linear transformation that maps $\mathbb{R}^d \to \mathbb{R}^m$.

$$
y = Wx
$$


In [ ]:
W = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
x = torch.tensor([1., 0.5, -1.])
y = W @ x
print('W shape:', W.shape)
print('x shape:', x.shape)
print('y = W@x:', y)


---

## 3) The Core Neural Network Layer: Affine Map

A standard layer is:

$$
y = Wx + b
$$

This is an **affine transformation** (linear + shift). Nearly everything in deep learning is built from this.


In [ ]:
b = torch.tensor([0.1, -0.2])
y = W @ x + b
print('y = W@x + b:', y)


---

## 4) Activation Functions (Nonlinearity)

Without nonlinearities, stacking layers collapses into a single linear map.

Common activations:

**ReLU**
$$
\mathrm{ReLU}(z) = \max(0, z)
$$

**Sigmoid**
$$
\sigma(z) = \frac{1}{1+e^{-z}}
$$

**Tanh**
$$
\tanh(z) = \frac{e^{z}-e^{-z}}{e^{z}+e^{-z}}
$$


In [ ]:
z = torch.linspace(-3, 3, 7)
print('z:', z)
print('ReLU(z):', F.relu(z))
print('sigmoid(z):', torch.sigmoid(z))
print('tanh(z):', torch.tanh(z))


---

## 5) Probability: Modeling $p(y\mid x)$

Many ML models output a conditional distribution:

$$
p(y\mid x; \theta)
$$

Read as: “probability of $y$ given $x$ under parameters $\theta$.”

For classification with $K$ classes, you’ll often see:

$$
p(y=k\mid x;\theta), \quad k \in \{1,\dots,K\}
$$


---

## 6) Softmax: Turning Scores (Logits) into Probabilities

Neural nets often output **logits** $z \in \mathbb{R}^K$. Softmax converts logits to probabilities:

$$
\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

Properties:
- $\mathrm{softmax}(z)_i \in (0,1)$
- $\sum_i \mathrm{softmax}(z)_i = 1$


In [ ]:
logits = torch.tensor([2.3, 1.1, 0.5])
probs = torch.softmax(logits, dim=0)
print('logits:', logits)
print('probs:', probs)
print('sum(probs):', probs.sum())


---

## 7) Loss Functions: Defining What “Good” Means

A loss is a scalar objective we want to minimize:

$$
L(\theta)
$$

For classification, a common choice is **cross-entropy**.


### Cross-Entropy (Multiclass)

If $y$ is one-hot and $p$ is predicted probability:

$$
L = -\sum_{i=1}^{K} y_i \log(p_i)
$$

If the true class is $c$, this simplifies to:

$$
L = -\log(p_c)
$$


In [ ]:
logits = torch.tensor([[2.3, 1.1, 0.5]])
true_class = torch.tensor([0])
loss = F.cross_entropy(logits, true_class)
print('cross-entropy loss:', loss.item())


---

## 8) Gradients: Sensitivity of Loss to Parameters

A partial derivative like:

$$
\frac{\partial L}{\partial w}
$$

means: “How much does the loss change if weight $w$ changes a little?”

For all parameters together, we use the gradient:

$$
\nabla_{\theta} L(\theta)
$$


In [ ]:
w = torch.tensor(1.5, requires_grad=True)
x = torch.tensor(2.0)
y_true = torch.tensor(5.0)

# simple model: y = w*x
# loss: (y - y_true)^2
loss = (w*x - y_true)**2
loss.backward()
print('loss:', loss.item())
print('dL/dw:', w.grad.item())


---

## 9) Chain Rule: The Math Behind Backpropagation

If a quantity depends on an intermediate variable:

$$
y = f(g(x))
$$

then the chain rule says:

$$
\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}
$$

Backpropagation is just applying this repeatedly through many layers.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
# y = (2x + 1)^2
y = (2*x + 1)**2
y.backward()
print('y:', y.item())
print('dy/dx:', x.grad.item())


---

## 10) Optimization: Gradient Descent Updates

The standard learning rule is:

$$
\theta \leftarrow \theta - \eta \nabla_{\theta} L(\theta)
$$

where $\eta$ is the learning rate.

**Interpretation:** move parameters in the direction that reduces the loss.


In [ ]:
w = torch.tensor(0.0, requires_grad=True)
x = torch.tensor(2.0)
y_true = torch.tensor(4.0)
eta = 0.1

for step in range(10):
    loss = (w*x - y_true)**2
    loss.backward()
    with torch.no_grad():
        w -= eta * w.grad
    w.grad.zero_()

print('learned w:', w.item())
print('expected w ~', (y_true/x).item())


---

## 11) Expectation: Averaging over Randomness / Data

Many objectives are written as expected values:

$$
\mathbb{E}_{x \sim p(x)}[f(x)]
$$

Read as: “average of $f(x)$ when $x$ is drawn from $p(x)$.”

In practice, we approximate expectations with sample averages:

$$
\mathbb{E}[f(x)] \approx \frac{1}{N}\sum_{n=1}^{N} f(x^{(n)})
$$


In [ ]:
samples = torch.randn(1000)
# Estimate E[X^2] for X~N(0,1). True value is 1.
est = (samples**2).mean()
print('Estimated E[X^2]:', est.item())


---

## 12) Regularization: Preventing Overfitting

A common pattern is adding a penalty term to the loss.

### L2 (Weight Decay)

$$
L_{\text{total}} = L + \lambda \lVert W \rVert_2^2
$$

Meaning: discourage large weights.

In practice, this is often implemented as **weight decay** in the optimizer.


In [ ]:
model = torch.nn.Linear(3, 2)
opt_no_decay = torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=0.0)
opt_decay    = torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-2)
print('Created two optimizers: one with weight_decay=0, one with weight_decay=1e-2')


---

## 13) KL Divergence: Comparing Distributions

KL divergence measures how one distribution differs from another:

$$
D_{\mathrm{KL}}(P\,\|\,Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)}
$$

In continuous form:

$$
D_{\mathrm{KL}}(P\,\|\,Q) = \int p(x)\log\frac{p(x)}{q(x)}\,dx
$$

You’ll see this constantly in:
- variational inference / VAEs
- diffusion model derivations
- distillation and information-theoretic objectives


In [ ]:
P = torch.tensor([0.7, 0.2, 0.1])
Q = torch.tensor([0.6, 0.3, 0.1])
kl = (P * (P.log() - Q.log())).sum()
print('KL(P||Q) =', kl.item())


---

## The Core Pattern Behind Many Papers

Most deep learning papers can be reduced to:

1. Represent input as vectors: $x \in \mathbb{R}^d$  
2. Transform through layers: $h = \phi(Wx + b)$  
3. Predict probabilities: $p(y\mid x;\theta)$  
4. Define a loss: $L(\theta)$  
5. Optimize: $\theta \leftarrow \theta - \eta \nabla_\theta L$  

Once these symbols feel familiar, papers stop feeling like a wall of math.
